In [1]:
from datasets import load_from_disk

ds = load_from_disk("../musiccaps/dataset_audio")
len(ds)

# preliminary results
# framework direkt metric output edicek
# machine translation metrics, BLEU, ROUGE, METEOR, CIDEr, bertscore

Loading dataset from disk:   0%|          | 0/19 [00:00<?, ?it/s]

5305

In [2]:
# First split: separate test set (held out for final evaluation only)
ds_split = ds.train_test_split(test_size=0.1, seed=42)
ds_train_val = ds_split["train"]
ds_test = ds_split["test"]

# Second split: split train+val into train and validation sets
ds_train_val_split = ds_train_val.train_test_split(test_size=0.1, seed=42)
ds_train = ds_train_val_split["train"]
ds_val = ds_train_val_split["test"]

print(f"Train: {len(ds_train)}, Val: {len(ds_val)}, Test: {len(ds_test)}")

Train: 4296, Val: 478, Test: 531


In [3]:
# Cell 1: Imports
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import ClapProcessor, ClapModel
import librosa
import torch
import torch.nn as nn
from pathlib import Path

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [4]:

# Constants (must match training)
D_AUDIO = 512  # CLAP projection dim
D_LM = 768     # GPT-2 embedding dim
PREFIX_LEN = 16  # Must match training


In [5]:

# Cell 3: Load Models
# CLAP (frozen)
clap_processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")
clap = ClapModel.from_pretrained("laion/clap-htsat-unfused").to(device)
clap.eval()
for p in clap.parameters():
    p.requires_grad = False

# GPT-2
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

gpt2 = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
gpt2.eval()
for p in gpt2.parameters():
    p.requires_grad = False


/home/aliozkaya/miniconda3/envs/musicgen/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [6]:

# Cell 4: Define Projection Networks (must match training architecture)
projection = nn.Sequential(
    nn.Linear(D_AUDIO, D_LM * 2),      # 512 → 1536
    nn.LayerNorm(D_LM * 2),
    nn.GELU(),
    nn.Dropout(0.35),
    nn.Linear(D_LM * 2, D_LM * 4),     # 1536 → 3072
    nn.LayerNorm(D_LM * 4),
    nn.GELU(),
    nn.Dropout(0.35),
    nn.Linear(D_LM * 4, PREFIX_LEN * D_LM),  # 3072 → PREFIX_LEN * 768
).to(device)

text_projection = nn.Sequential(
    nn.Linear(D_LM, D_AUDIO),
    nn.LayerNorm(D_AUDIO),
    nn.Dropout(0.5),
    nn.GELU(),
).to(device)


In [7]:
# Cell 5: Load Checkpoint
SAVE_DIR = Path("../04_train/checkpoints")  # Adjust path as needed

checkpoint = torch.load(SAVE_DIR / "best_model_stage2.pt")

projection.load_state_dict(checkpoint["projection"])
text_projection.load_state_dict(checkpoint["text_projection"])
gpt2.load_state_dict(checkpoint["gpt2"])  # Load fine-tuned GPT-2

print("Model loaded successfully!")

Model loaded successfully!


In [8]:
# Cell 6: Helper Functions
def get_audio_embedding(sample):
    """Get audio embedding for a single sample"""
    audio = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]

    if audio.ndim == 2:
        audio = audio.mean(axis=1)

    if sr != 48000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=48000)

    inputs = clap_processor(
        audios=audio,
        sampling_rate=48000,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        emb = clap.get_audio_features(**inputs)

    return emb  # (1, D_AUDIO)


In [9]:
# Cell 7: Generation Function
def generate_caption(audio_sample, use_beam_search=True, max_length=100, min_length=20):
    """
    Generate caption from audio sample.
    
    Args:
        audio_sample: Dictionary with 'audio' key containing 'array' and 'sampling_rate'
        use_beam_search: If True, use beam search (better quality). If False, use sampling.
        max_length: Maximum length of generated caption (includes prefix)
        min_length: Minimum length of generated caption (includes prefix)
    
    Returns:
        Generated caption string
    """
    # Get audio embedding
    audio_emb = get_audio_embedding(audio_sample)
    
    # Project to prefix tokens
    prefix = projection(audio_emb)
    prefix = prefix.view(1, PREFIX_LEN, D_LM)
    
    # Get EOS token ID
    eos_token_id = tokenizer.eos_token_id
    pad_token_id = tokenizer.pad_token_id
    
    # Generate caption
    if use_beam_search:
        generated = gpt2.generate(
            inputs_embeds=prefix,
            max_length=max_length,
            min_length=min_length,
            num_beams=10,  # Balanced between quality and speed
            early_stopping=True,
            repetition_penalty=2.25,  # High penalty to prevent repetition
            no_repeat_ngram_size=2,  # Prevent 2-gram repetition (catches "passionate and passionate")
            length_penalty=0.8,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
            do_sample=False,  # Deterministic with beam search
        )
    else:
        # Alternative: sampling with lower temperature
        generated = gpt2.generate(
            inputs_embeds=prefix,
            max_length=max_length,
            min_length=min_length,
            do_sample=True,
            temperature=0.3,  # Increased from 0.1 for more natural endings
            top_k=20,
            top_p=0.8,
            repetition_penalty=1.8,  # Increased to prevent repetition
            no_repeat_ngram_size=2,  # Prevent 2-gram repetition
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
        )
    
    # Decode to text - stop at EOS token
    caption = tokenizer.decode(generated[0], skip_special_tokens=True)
    
    # Additional cleanup: remove incomplete sentences at the end
    # Find last complete sentence (ends with . ! ?)
    last_period = max(caption.rfind('.'), caption.rfind('!'), caption.rfind('?'))
    if last_period > len(caption) * 0.5:  # Only if sentence is substantial
        caption = caption[:last_period + 1]
    
    return caption



In [10]:
# Cell 8: SPIDER Metric Evaluation Setup
import nltk
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from collections import Counter
import numpy as np

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('wordnet', quiet=True)
    nltk.download('omw-1.4', quiet=True)

print("SPIDER metric libraries loaded successfully!")


SPIDER metric libraries loaded successfully!


In [11]:
# Cell 9: SPIDER Metric Computation Functions
def compute_spider_metric(generated_captions, ground_truth_captions):
    """
    Compute SPIDER metric (average of normalized BLEU-4, METEOR, ROUGE-L, CIDEr).
    
    Args:
        generated_captions: List of generated caption strings
        ground_truth_captions: List of ground truth caption strings
    
    Returns:
        Dictionary with individual metrics and SPIDER score
    """
    smooth = SmoothingFunction().method1
    rouge_scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    
    bleu_scores = []
    meteor_scores = []
    rouge_scores = []
    cider_scores = []
    
    print("Computing metrics for each caption pair...")
    for i, (gen, gt) in enumerate(zip(generated_captions, ground_truth_captions)):
        # Tokenize
        gen_tokens = gen.lower().split()
        gt_tokens = gt.lower().split()
        
        # Skip if empty
        if len(gen_tokens) == 0 or len(gt_tokens) == 0:
            bleu_scores.append(0.0)
            meteor_scores.append(0.0)
            rouge_scores.append(0.0)
            cider_scores.append(0.0)
            continue
        
        # BLEU-4
        try:
            bleu = sentence_bleu([gt_tokens], gen_tokens, smoothing_function=smooth)
            bleu_scores.append(bleu)
        except:
            bleu_scores.append(0.0)
        
        # METEOR
        try:
            meteor = meteor_score([gt_tokens], gen_tokens)
            meteor_scores.append(meteor)
        except Exception as e:
            meteor_scores.append(0.0)
        
        # ROUGE-L
        try:
            rouge_scores_dict = rouge_scorer_obj.score(gt, gen)
            rouge_scores.append(rouge_scores_dict['rougeL'].fmeasure)
        except:
            rouge_scores.append(0.0)
        
        # CIDEr approximation (TF-IDF based similarity)
        # This is a simplified version - full CIDEr is more complex
        gen_words = set(gen_tokens)
        gt_words = set(gt_tokens)
        if len(gen_words) > 0 and len(gt_words) > 0:
            # Jaccard similarity as CIDEr approximation
            intersection = len(gen_words & gt_words)
            union = len(gen_words | gt_words)
            cider = intersection / union if union > 0 else 0.0
        else:
            cider = 0.0
        cider_scores.append(cider)
        
        if (i + 1) % 50 == 0:
            print(f"  Processed {i + 1}/{len(generated_captions)} samples...")
    
    # Average scores
    avg_bleu = np.mean(bleu_scores)
    avg_meteor = np.mean(meteor_scores)
    avg_rouge = np.mean(rouge_scores)
    avg_cider = np.mean(cider_scores)
    
    # Normalize CIDEr (simple approximation - full CIDEr can be much higher)
    # Scale to [0, 1] range (CIDEr typically ranges 0-10+, we normalize by assuming max ~2.0)
    normalized_cider = min(avg_cider * 0.5, 1.0)
    
    # SPIDER = average of normalized scores
    spider = (avg_bleu + avg_meteor + avg_rouge + normalized_cider) / 4.0
    
    return {
        'BLEU_4': avg_bleu,
        'METEOR': avg_meteor,
        'ROUGE_L': avg_rouge,
        'CIDEr': normalized_cider,
        'CIDEr_raw': avg_cider,  # Raw CIDEr for reference
        'SPIDER': spider,
        'individual_scores': {
            'bleu': bleu_scores,
            'meteor': meteor_scores,
            'rouge': rouge_scores,
            'cider': cider_scores
        }
    }


In [12]:
# Cell 10: Evaluation Function
def evaluate_model(dataset, num_samples=None, use_beam_search=True, max_length=100, min_length=20):
    """
    Evaluate model on dataset and compute SPIDER metric.
    
    Args:
        dataset: Test dataset
        num_samples: Number of samples to evaluate (None = all)
        use_beam_search: Whether to use beam search for generation
        max_length: Maximum generation length
        min_length: Minimum generation length
    
    Returns:
        Dictionary with metrics, generated captions, and ground truth captions
    """
    if num_samples is None:
        num_samples = len(dataset)
    else:
        num_samples = min(num_samples, len(dataset))
    
    generated_captions = []
    ground_truth_captions = []
    
    print(f"Generating captions for {num_samples} samples...")
    print("=" * 60)
    
    for idx in range(num_samples):
        sample = dataset[idx]
        try:
            generated = generate_caption(
                sample, 
                use_beam_search=use_beam_search,
                max_length=max_length,
                min_length=min_length
            )
            generated_captions.append(generated)
            ground_truth_captions.append(sample["caption"])
        except Exception as e:
            print(f"Error processing sample {idx}: {e}")
            continue
        
        if (idx + 1) % 50 == 0:
            print(f"  Generated {idx + 1}/{num_samples} captions...")
    
    print(f"\nGenerated {len(generated_captions)} captions successfully.")
    print("Computing SPIDER metric...")
    print("=" * 60)
    
    # Compute metrics
    metrics = compute_spider_metric(generated_captions, ground_truth_captions)
    
    # Print results
    print("\n" + "=" * 60)
    print("EVALUATION RESULTS")
    print("=" * 60)
    print(f"BLEU-4:    {metrics['BLEU_4']:.4f}")
    print(f"METEOR:    {metrics['METEOR']:.4f}")
    print(f"ROUGE-L:   {metrics['ROUGE_L']:.4f}")
    print(f"CIDEr:     {metrics['CIDEr']:.4f} (raw: {metrics['CIDEr_raw']:.4f})")
    print(f"SPIDER:    {metrics['SPIDER']:.4f}")
    print("=" * 60)
    
    return {
        'metrics': metrics,
        'generated_captions': generated_captions,
        'ground_truth_captions': ground_truth_captions
    }


In [ ]:
# Cell 11: Run Evaluation on Test Set
# Evaluate on all test samples (or specify num_samples for quick test)
results = evaluate_model(
    ds_test, 
    num_samples=None,  # Set to None for full evaluation, or e.g., 100 for quick test
    use_beam_search=True,
    max_length=80,
    min_length=20
)


Generating captions for 531 samples...
  Generated 50/531 captions...
  Generated 100/531 captions...
  Generated 150/531 captions...
  Generated 200/531 captions...
  Generated 250/531 captions...


In [ ]:
# Cell 12: Display Sample Results
# Show some example generated vs ground truth captions
num_examples = 5
print(f"\nSample Results (first {num_examples}):")
print("=" * 60)

for i in range(min(num_examples, len(results['generated_captions']))):
    print(f"\nSample {i+1}:")
    print(f"Generated:  {results['generated_captions'][i]}")
    print(f"Ground truth: {results['ground_truth_captions'][i]}")
    print("-" * 60)


In [ ]:
# Cell 13: Save Results (Optional)
import json
from datetime import datetime

# Save metrics to file
results_to_save = {
    'timestamp': datetime.now().isoformat(),
    'num_samples': len(results['generated_captions']),
    'metrics': {
        'BLEU_4': float(results['metrics']['BLEU_4']),
        'METEOR': float(results['metrics']['METEOR']),
        'ROUGE_L': float(results['metrics']['ROUGE_L']),
        'CIDEr': float(results['metrics']['CIDEr']),
        'SPIDER': float(results['metrics']['SPIDER']),
    },
    'config': {
        'use_beam_search': True,
        'max_length': 100,
        'min_length': 20,
    }
}

# Save to JSON
output_file = Path("evaluation_results.json")
with open(output_file, 'w') as f:
    json.dump(results_to_save, f, indent=2)

print(f"\nResults saved to {output_file}")
print(f"SPIDER Score: {results['metrics']['SPIDER']:.4f}")
